In [2]:
# 1. 필수 라이브러리 설치 (langchain-community 추가)
!pip install -q langchain-chroma langchain-huggingface sentence-transformers pypdf langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.37.0 requires opentelemetry-exporter-otlp-proto-common==1.37.0, but you have opentelemetry-exporter-otlp-proto-common 1.38.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.37.0 requires opentelemetry-proto==1.37.0, but you have opentelemetry-proto 1.38.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.37.0 requires opentelemetry-sdk~=1.37.0,

In [6]:

import os
import shutil
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from google.colab import files

# ==========================================
# 2. 임베딩 모델 로드 (jhgan/ko-sroberta-multitask)
# ==========================================
print("--- [1/5] 임베딩 모델 로드 중... ---")
model_name = "jhgan/ko-sroberta-multitask"
model_kwargs = {'device': 'cpu'} # Colab GPU 사용 시 'cuda'로 변경 가능
encode_kwargs = {'normalize_embeddings': True} # 코사인 유사도 성능 향상

hf_embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

# ==========================================
# 3. PDF 파일 로드 및 텍스트 추출
# ==========================================
print("--- [2/5] PDF 파일 읽는 중... ---")

# 업로드하신 파일명과 정확히 일치해야 합니다.
file_name = "/content/(이용약관전문)인터넷서비스이용약관_202509.pdf"

if not os.path.exists(file_name):
    print(f"❌ 오류: '{file_name}' 파일이 없습니다. Colab 파일 탭에 업로드해주세요.")
else:
    loader = PyPDFLoader(file_name)
    # 페이지 단위로 문서를 가져옵니다 (메타데이터에 page 번호 자동 포함됨)
    documents = loader.load()
    print(f"✅ 총 {len(documents)} 페이지를 로드했습니다.")

    # ==========================================
    # 4. 텍스트 분할 (Chunking) - 개선된 크기 적용
    # ==========================================
    print("--- [3/5] 문서 분할 중 (Chunk Size: 600)... ---")

    # 약관 조항이 잘리지 않도록 크기를 600으로 늘림
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=600,
        chunk_overlap=100, # 문맥이 끊기지 않도록 겹치는 구간 설정
        separators=["\n\n", "\n", ".", " ", ""] # 문단 -> 줄바꿈 -> 마침표 순으로 자름
    )

    splits = text_splitter.split_documents(documents)
    print(f"✅ 총 {len(splits)}개의 검색 가능한 조각(Chunk)으로 분할되었습니다.")

    # ==========================================
    # 5. 벡터 DB 생성 및 저장
    # ==========================================
    persist_directory = "./chroma_db_kt_terms"

    # 기존 DB가 있다면 삭제하고 새로 생성 (중복 방지)
    if os.path.exists(persist_directory):
        shutil.rmtree(persist_directory)

    print(f"--- [4/5] 벡터 DB 생성 및 저장 중 ({persist_directory})... ---")

    vectorstore = Chroma.from_documents(
        documents=splits,
        embedding=hf_embeddings,
        persist_directory=persist_directory,
        collection_name="kt_terms"
    )

    print("✅ 벡터 DB 생성이 완료되었습니다!")

    # ==========================================
    # 6. 검색 테스트 (검증)
    # ==========================================
    print("\n--- [5/5] 검색 테스트 ---")
    query = "해지 위약금은 어떻게 계산하나요?" # 실제 약관에 있을법한 질문

    # 유사도 점수도 같이 보기 (score가 낮을수록 유사함, L2 distance 기준)
    # cosine 유사도일 경우 높을수록 유사함 (Chroma 설정에 따라 다름)
    results = vectorstore.similarity_search(query, k=3)

    print(f"🔍 질문: '{query}'")
    print("-" * 50)
    for i, doc in enumerate(results):
        print(f"[결과 {i+1}] (페이지: {doc.metadata.get('page', 0) + 1}쪽)")
        # PyPDFLoader는 0부터 시작하므로 +1 해줌
        print(f"내용: {doc.page_content[:150]}...") # 앞 150자만 미리보기
        print("-" * 50)

    # ==========================================
    # 7. 다운로드 (압축)
    # ==========================================
    print("\n📦 DB 폴더 압축 중...")
    shutil.make_archive("chroma_db_kt_terms", 'zip', persist_directory)
    print("✅ 압축 완료. 다운로드를 시작합니다.")

    files.download('chroma_db_kt_terms.zip')

--- [1/5] 임베딩 모델 로드 중... ---
--- [2/5] PDF 파일 읽는 중... ---
✅ 총 232 페이지를 로드했습니다.
--- [3/5] 문서 분할 중 (Chunk Size: 600)... ---
✅ 총 450개의 검색 가능한 조각(Chunk)으로 분할되었습니다.
--- [4/5] 벡터 DB 생성 및 저장 중 (./chroma_db_kt_terms)... ---
✅ 벡터 DB 생성이 완료되었습니다!

--- [5/5] 검색 테스트 ---
🔍 질문: '해지 위약금은 어떻게 계산하나요?'
--------------------------------------------------
[결과 1] (페이지: 167쪽)
내용: -  167 - 
 
※ 계약기간 이내 AP를 해지하거나 계약기간을 단축할 시는 추가 할인된 요금을 반환해야 하며, 
   할인반환금은 아래의 산식을 적용 받습니다. 
   - 산정식 : ∑ [약정기간별 총 할인금액 X (1 – 약정기간 별 할인반환금 할인율) 
   ...
--------------------------------------------------
[결과 2] (페이지: 116쪽)
내용: ※ 계약기간 이내 해지시는 할인받은 요금 및 단말장치사용료를 반환하여야 합니다. 
  - 이용료 산정식 = 할인금액 ⅹ 경과월수 ⅹ(1-사용기간 할인율/계약기간 할인율) 
   · 사용기간 할인율은 2년 미만은 1년 계약, 3년 미만은 2년 계약, 4년 미만은 3년 계...
--------------------------------------------------
[결과 3] (페이지: 208쪽)
내용: -  208 - 
 
 
종 별 구분 이용요금 비 고 
OFFICE 신규 110,000원 
3년이상 약정시 면제 
(단, 3년이내 해지시 할인반환금 산정식에 따라 부과) 
“멀티넷” 
기본 
(신규/이전) 
110,000원 
- 신규 가입, 이전 시 출동 및 소요 비용...
------------------------------

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>